# 05 Tree Baselines (Local-only + Centralized)

This notebook runs reviewer-requested tree baselines for direct comparison with MLP/LSTM:
- Local-only XGBoost
- Centralized global XGBoost
- Local-only LightGBM
- Centralized global LightGBM

Outputs are saved to `saved_models/` as CSV files.

In [9]:
# Optional install (run once if needed)
# !pip install xgboost lightgbm

import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

DATA_DIR = Path('ashrae')
SAVE_DIR = Path('saved_models')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.cleaned_isamu_matt.building_mean_y_ge_1.csv'
WEATHER_PATH = DATA_DIR / 'weather_train.cleaned.csv'
META_PATH = DATA_DIR / 'building_metadata.csv'

assert TRAIN_PATH.exists(), f'Missing file: {TRAIN_PATH}'
assert WEATHER_PATH.exists(), f'Missing file: {WEATHER_PATH}'
assert META_PATH.exists(), f'Missing file: {META_PATH}'


In [10]:
# Load data
train_raw = pd.read_csv(TRAIN_PATH)
weather = pd.read_csv(WEATHER_PATH)
meta = pd.read_csv(META_PATH)

for d in (train_raw, weather):
    if 'timestamp' in d.columns:
        d['timestamp'] = pd.to_datetime(d['timestamp'])

# Merge metadata without duplicating columns already present in cleaned train
meta_extra_cols = [c for c in meta.columns if c not in train_raw.columns]
if meta_extra_cols:
    df = train_raw.merge(meta[['building_id'] + meta_extra_cols], on='building_id', how='left')
else:
    df = train_raw.copy()

if 'site_id' not in df.columns:
    raise KeyError('site_id is missing after metadata merge. Check train_raw/meta columns.')
df = df.merge(weather, on=['site_id', 'timestamp'], how='left')

# Keep electric meter only when meter exists
if 'meter' in df.columns:
    df = df[df['meter'] == 0].copy()

# Encode primary_use
if 'primary_use' in df.columns:
    df['primary_use_enc'] = pd.factorize(df['primary_use'])[0]
else:
    df['primary_use_enc'] = 0

# Ensure target
if 'meter_reading' not in df.columns:
    raise RuntimeError('Expected meter_reading column is missing.')

df['meter_reading'] = df['meter_reading'].clip(lower=0)
df['y'] = np.log1p(df['meter_reading'])

print('Rows after merge/filter:', len(df))
print('Buildings:', df['building_id'].nunique())

Rows after merge/filter: 11362529
Buildings: 1377


In [11]:
# Feature engineering aligned with MLP/LSTM setup
df['hour'] = df['timestamp'].dt.hour
df['dow'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month
df['is_weekend'] = (df['dow'] >= 5).astype(int)

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['dow'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dow'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df['log_sqft'] = np.log1p(df['square_feet'].fillna(df['square_feet'].median()))
df['year_built'] = df['year_built'].fillna(df['year_built'].median())
df['building_age'] = 2017 - df['year_built']
df['floor_count'] = df['floor_count'].fillna(1)

weather_cols = [
    'air_temperature', 'dew_temperature', 'wind_speed', 'cloud_coverage',
    'precip_depth_1_hr', 'sea_level_pressure'
]
for c in weather_cols:
    if c not in df.columns:
        df[c] = 0.0
    df[c] = df[c].fillna(df[c].median())

# Lags per building
df = df.sort_values(['building_id', 'timestamp']).copy()
g = df.groupby('building_id')['y']
df['lag_1'] = g.shift(1)
df['lag_24'] = g.shift(24)
df['roll24_mean'] = g.shift(1).rolling(24).mean().reset_index(level=0, drop=True)
df['roll24_std'] = g.shift(1).rolling(24).std().reset_index(level=0, drop=True)

for c in ['lag_1', 'lag_24', 'roll24_mean', 'roll24_std']:
    df[c] = df[c].fillna(df[c].median())

FEATURES = [
    'site_id', 'building_id', 'primary_use_enc',
    'hour', 'dow', 'month', 'is_weekend',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'log_sqft', 'building_age', 'floor_count',
    'lag_1', 'lag_24', 'roll24_mean', 'roll24_std',
] + weather_cols

TARGET = 'y'

print('Features:', len(FEATURES))

Features: 26


In [12]:
# Build per-building train/val/test client split
TRAIN_FRAC = 0.75
VAL_FRAC = 0.10
MIN_ROWS_PER_BUILDING = 2000
N_CLIENTS = 60
NON_IID_BY_SITE = True
SITES_TO_USE = 6

all_buildings = df['building_id'].unique()
if NON_IID_BY_SITE:
    site_counts = df.groupby('site_id')['building_id'].nunique().sort_values(ascending=False)
    top_sites = site_counts.head(SITES_TO_USE).index.tolist()
    candidate_buildings = df[df['site_id'].isin(top_sites)]['building_id'].unique()
    chosen_buildings = np.random.choice(candidate_buildings, size=min(N_CLIENTS, len(candidate_buildings)), replace=False)
else:
    chosen_buildings = np.random.choice(all_buildings, size=min(N_CLIENTS, len(all_buildings)), replace=False)

chosen_buildings = sorted(chosen_buildings.tolist())
client_data = {}

def time_split_building(d):
    d = d.sort_values('timestamp').reset_index(drop=True)
    n = len(d)
    n_train = int(n * TRAIN_FRAC)
    n_val = int(n * VAL_FRAC)
    tr = d.iloc[:n_train].copy()
    va = d.iloc[n_train:n_train + n_val].copy()
    te = d.iloc[n_train + n_val:].copy()
    return tr, va, te

for bid in chosen_buildings:
    d = df[df['building_id'] == bid].copy()
    if len(d) < MIN_ROWS_PER_BUILDING:
        continue
    tr, va, te = time_split_building(d)
    if min(len(tr), len(va), len(te)) < 50:
        continue
    client_data[bid] = {'train': tr, 'val': va, 'test': te}

usable_bids = sorted(client_data.keys())
print('Usable clients:', len(usable_bids))

Usable clients: 60


In [13]:
# Cold-start setup + save helper
ENABLE_COLD_START = True
COLD_START_RATIO = 0.20
COLD_TRAIN_DAYS = 14
MIN_TRAIN_ROWS_COLD = 300

if ENABLE_COLD_START and len(usable_bids) > 0:
    cold_set = set(np.random.choice(usable_bids, size=max(1, int(COLD_START_RATIO * len(usable_bids))), replace=False))
    for bid in cold_set:
        tr = client_data[bid]['train'].copy()
        start = tr['timestamp'].min()
        tr_cs = tr[tr['timestamp'] < (start + pd.Timedelta(days=COLD_TRAIN_DAYS))].copy()
        if len(tr_cs) < MIN_TRAIN_ROWS_COLD:
            tr_cs = tr.iloc[:MIN_TRAIN_ROWS_COLD].copy()
        client_data[bid]['train_orig'] = tr.copy()
        client_data[bid]['train'] = tr_cs
else:
    cold_set = set()

source_bids = [b for b in usable_bids if b not in cold_set]

def save_dataframe(df, name):
    out = SAVE_DIR / f'{name}.csv'
    df.to_csv(out, index=False)
    print(f'Saved: {out}')

print('Cold-start clients:', len(cold_set))
print('Warm/source clients:', len(source_bids))

Cold-start clients: 12
Warm/source clients: 48


In [14]:
# Run reviewer-requested tree baselines
import importlib
import run_tree_baselines as tree_baselines

tree_baselines = importlib.reload(tree_baselines)
run_tree_baselines = tree_baselines.run_tree_baselines

tree_results = run_tree_baselines(
    seed=SEED,
    client_data=client_data,
    usable_bids=usable_bids,
    target=TARGET,
    features=FEATURES,
    cold_set=cold_set,
    source_bids=source_bids,
)
globals().update(tree_results)

print('\nSummary table:')
df_tree_baseline_summary

Tree Baselines - XGBoost
Saved: saved_models\tree_centralized_xgb.csv
Saved: saved_models\tree_local_only_xgb.csv

Centralized XGBoost
  Mean MAE (log1p): 0.0903
  Mean RMSE (log1p): 0.1548
  Mean CVRMSE (raw %): 10.94%
  Mean WAPE (raw %): 7.59%
  Cold-start MAE: 0.1418
  Cold-start CVRMSE: 13.55%

Local-only XGBoost
  Mean MAE (log1p): 0.1461
  Mean RMSE (log1p): 0.2553
  Mean CVRMSE (raw %): 15.72%
  Mean WAPE (raw %): 11.20%
  Cold-start MAE: 0.2162
  Cold-start CVRMSE: 20.79%

Tree Baselines - LightGBM
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007906 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2362
[LightGBM] [Info] Number of data points in the train set: 305655, number of used features: 25
[LightGBM] [Info] Start training from score 4.248756
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testin

,Strategy,Overall,Warm,Cold,CVRMSE (%)
2,C. Centralized-LGBM,0.071724,0.070584,0.076287,9.943927
0,C. Centralized-XGB,0.090338,0.077474,0.141794,10.942152
3,0. Local-only-LGBM,0.134685,0.103520,0.259348,14.366808
1,0. Local-only-XGB,0.146110,0.128587,0.216205,15.720108


In [15]:
# Optional: compare tree baselines vs saved local-only MLP/LSTM files
from pathlib import Path

rows = []

if 'df_local_xgb' in globals():
    rows.append({'Model': 'Local-only XGBoost', 'MAE (log1p)': float(df_local_xgb['mae'].mean())})
if 'df_central_xgb' in globals():
    rows.append({'Model': 'Centralized XGBoost', 'MAE (log1p)': float(df_central_xgb['mae'].mean())})
if 'df_local_lgbm' in globals():
    rows.append({'Model': 'Local-only LightGBM', 'MAE (log1p)': float(df_local_lgbm['mae'].mean())})
if 'df_central_lgbm' in globals():
    rows.append({'Model': 'Centralized LightGBM', 'MAE (log1p)': float(df_central_lgbm['mae'].mean())})

mlp_path = Path('saved_models/strategy0_local_only_results.csv')
lstm_path = Path('saved_models/strategy0_local_only_lstm.csv')

if mlp_path.exists():
    d = pd.read_csv(mlp_path)
    if 'mae' in d.columns:
        rows.append({'Model': 'Local-only MLP (saved)', 'MAE (log1p)': float(d['mae'].mean())})

if lstm_path.exists():
    d = pd.read_csv(lstm_path)
    if 'mae' in d.columns:
        rows.append({'Model': 'Local-only LSTM (saved)', 'MAE (log1p)': float(d['mae'].mean())})

compare_df = pd.DataFrame(rows).sort_values('MAE (log1p)', ascending=True).reset_index(drop=True)
compare_df

,Model,MAE (log1p)
0,Centralized LightGBM,0.071724
1,Centralized XGBoost,0.090338
2,Local-only LightGBM,0.134685
3,Local-only XGBoost,0.146110
4,Local-only LSTM (saved),0.340393
5,Local-only MLP (saved),0.797665
